# 10 — Particionamento e Shuffle

**EP · Data Science Knowledge Base | Big Data e Processamento Distribuído | Notebook 10/16**

**Execução:** use o kernel Python 3, reinicie o kernel e execute as células em ordem. Laboratórios Spark são opcionais e desativados por padrão.

**Abordagem:** os exemplos locais verificam conceitos e não representam um cluster distribuído.


## Objetivos de aprendizagem
Explicar por que operações corretas podem ser lentas em escala.

Ao final, você deve conseguir interpretar os resultados, justificar as escolhas e explicar suas limitações.

## Fundamentação teórica
Particionamento, skew, coalesce versus repartition, cache, collect e explain.

**Atenção metodológica:** execução local facilita entendimento e depuração, mas não comprova ganhos de paralelismo em infraestrutura distribuída. Escalabilidade exige medir dados, nós, rede, memória, partições e operações de shuffle.

In [1]:
from pathlib import Path
import sys
DATASETS = Path('datasets')
assert DATASETS.is_dir(), 'Execute a partir da raiz da área 10 (pasta datasets não encontrada)'
print('Python:', sys.version.split()[0])
print('Pasta dos dados:', DATASETS.as_posix())

Python: 3.14.4
Pasta dos dados: datasets


## Demonstração 1 — construção e observação

In [2]:
from collections import Counter
import pandas as pd
v=pd.read_csv('datasets/flights.csv')
print('Distribuição por companhia:',Counter(v['carrier']).most_common(8))
print('Maior participação:',round(v['carrier'].value_counts(normalize=True).max(),3))

Distribuição por companhia: [('AS', 3784), ('WN', 1394), ('OO', 1186), ('DL', 1082), ('UA', 1051), ('AA', 482), ('US', 367), ('B6', 214)]
Maior participação: 0.378


## Interpretação
Observe o esquema, as unidades e os resultados. Antes de concluir que um algoritmo é eficiente, distinga **complexidade computacional**, **custo de comunicação** e **volume de dados**. Os cálculos locais ilustram a semântica; o Spark, quando disponível, ilustra a API distribuída.

## Demonstração 2 — validação e limitações

In [3]:
# Simulação de skew determinística; não inicializa Spark.
from hashlib import sha256
pares = ['chave_quente']*900+[f'chave_{i}' for i in range(100)]
def balde(chave):
    return int.from_bytes(sha256(chave.encode('utf-8')).digest()[:4],'big') % 4
cargas = [sum(balde(chave)==j for chave in pares) for j in range(4)]
assert sum(cargas)==1000 and max(cargas)>=900
print('Partições por chave:',cargas)
print('Desbalanceamento máximo/média:',round(max(cargas)/(sum(cargas)/len(cargas)),2))
print('Comparação conceitual sem custo de shuffle ou medição de rede.')


Partições por chave: [26, 28, 18, 928]
Desbalanceamento máximo/média: 3.71
Comparação conceitual sem custo de shuffle ou medição de rede.


## Boas práticas e limitações

- Execute com versões registradas, datasets identificados e caminhos relativos.
- Diferencie execução local, emulação lógica e infraestrutura distribuída real.
- Documente nulos, chaves, cardinalidade, custos e limitações das métricas.
- Não publique datasets ou resultados cuja redistribuição não esteja autorizada.
- Evite `collect()` de grandes tabelas e não confunda uma execução `local[*]` com cluster multinó.


## Exercícios
1. Explique que parte do exemplo poderia ser distribuída por chave ou partição e qual é o custo do shuffle.
2. Descreva uma validação de qualidade para os dados usados.
3. Indique um cenário em que o exemplo local deixaria de ser adequado.

## Respostas comentadas (orientativas)
1. Agregações por chave podem combinar resultados parciais, mas exigem redistribuir dados com a mesma chave; filtros e projeções usualmente evitam shuffle.
2. Verifique contagem, unicidade de chaves, percentual de valores ausentes, tipos, intervalos e consistência das unidades.
3. Quando dados, latência, concorrência ou exigências de falha excederem a capacidade de uma máquina; nesse caso, considere Spark/Hadoop e meça os custos adicionais.

## Síntese

O capítulo 10 reuniu conceitos e verificações de **skew e movimentação de dados**. Os exemplos de pequena escala não comprovam desempenho em cluster. Consulte apenas referências complementares efetivamente publicadas e documentadas.


## Revisão científica — auditoria e aplicação

**Ponto de atenção deste capítulo:** partições e skew. Esta seção complementa os exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **partições e skew** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **partições e skew** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.


## Laboratório opcional — Spark local real

Requer `RUN_SPARK_LABS=1` e `SPARK_LAB_PYTHON` apontando para um ambiente isolado com PySpark e Java. O teste falha explicitamente se executado incorretamente; resultados locais não comprovam cluster distribuído.


In [4]:
# Laboratório Spark real opcional, executado em interpretador isolado.
# Ative SOMENTE quando o ambiente PySpark e o Java já estiverem preparados.
import os, sys, subprocess, textwrap
if os.environ.get('RUN_SPARK_LABS') == '1':
    interpretador = os.environ.get('SPARK_LAB_PYTHON', sys.executable)
    assert os.path.isfile(interpretador), f'Interpretador não encontrado: {interpretador}'
    programa = textwrap.dedent(r'''from pyspark.sql import SparkSession
s=SparkSession.builder.master('local[2]').appName('ep-area10-10').config('spark.ui.enabled','false').getOrCreate()
try:
    s.sparkContext.setLogLevel('ERROR')
    pares=[('chave_quente',1)]*900+[(f'chave_{i}',1) for i in range(100)]
    rdd=s.sparkContext.parallelize(pares,4).partitionBy(4)
    cargas=rdd.mapPartitions(lambda it:[sum(1 for _ in it)]).collect()
    assert sum(cargas)==1000 and max(cargas)>=900,cargas
    reducao=dict(rdd.reduceByKey(lambda a,b:a+b).collect())
    assert reducao['chave_quente']==900
    print('Carga por partição após partitionBy:',cargas)
    print('O skew é artificial; não representa benchmark de rede ou cluster.')
    print('SPARK_LAB_OK_10')
finally:s.stop()''')
    ambiente = os.environ.copy()
    ambiente.setdefault('SPARK_LOCAL_IP', '127.0.0.1')
    ambiente['PYSPARK_PYTHON'] = interpretador
    ambiente['PYSPARK_DRIVER_PYTHON'] = interpretador
    resultado = subprocess.run([interpretador, '-c', programa], cwd=os.getcwd(),
                               env=ambiente, text=True, capture_output=True, timeout=150)
    if resultado.stdout: print(resultado.stdout.strip())
    if resultado.returncode:
        print(resultado.stderr[-3500:])
        raise RuntimeError('O LABORATÓRIO SPARK 10 FALHOU; não registrar como aprovado.')
    assert 'SPARK_LAB_OK_10' in resultado.stdout, 'Marcador de sucesso Spark não encontrado'
else:
    print('SPARK NÃO EXECUTADO (opcional). Ative RUN_SPARK_LABS=1 e SPARK_LAB_PYTHON.')


Carga por partição após partitionBy: [24, 22, 29, 925]
O skew é artificial; não representa benchmark de rede ou cluster.
SPARK_LAB_OK_10
